# Data Source and Preparation

```mermaid
flowchart LR
    A[Time signal] --> B[Windowing]
    B --> C[Time-domain amplitude]
    C --> D[FFT]
    D --> E[Harmonic structure analysis]
    E --> F[Chatter decision]
```

The dataset consists of vibration signals recorded during industrial cutting experiments. Each experiment directory corresponds to a specific machining configuration (e.g., tool stick-out length, cutting parameters).

Raw measurements are stored as MATLAB .mat files and contain multi-channel time series data.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from scipy.signal import find_peaks

In [4]:
# Download and extract the original dataset from Mendeley (if not already present)
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

URL = "https://data.mendeley.com/public-files/datasets/hvm4wh3jzx/files/73bc3318-3e8e-4566-b565-e726a3d75188/file_downloaded"
RAW_ZIP = Path('../data/raw/mendeley_dataset.zip')
RAW_EXTRACT_DIR = Path('../data/raw_mat')
RAW_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_ZIP.exists():
    try:
        import requests
        RAW_ZIP.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(URL, stream=True) as r:
            r.raise_for_status()
            with open(RAW_ZIP, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
    except Exception:
        # fallback to urllib
        RAW_ZIP.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(URL, RAW_ZIP)

# Try to unzip; if not a zip, keep the downloaded file for inspection
try:
    with zipfile.ZipFile(RAW_ZIP, 'r') as z:
        z.extractall(RAW_EXTRACT_DIR)
    print('Extracted dataset to', RAW_EXTRACT_DIR)
except zipfile.BadZipFile:
    print('Downloaded file is not a ZIP archive. Saved to', RAW_ZIP)


Extracted dataset to ../data/raw_mat


## Configuration of input/output directory and data
- path to raw data
- output path for processed data
- data definition:
    - Accelerometer axes (x, y, z)
    - USE_CHANNEL: axis for labeling decision (see Turning_Cutting_Data_Description.pdf for labeling approach) --> shows dominant vibration direction in cutting process
- WINDOW_SEC: to transform the continous timeseries in samples, a fix-length sliding window is used; the ML-approach is not able to work on continous data, it needs fixed length windows to perform data preparation --> Provides sufficient temporal context for chatter development; 

        - Empirically chosen as trade-off between:
        - temporal resolution (shorter windows)
        - spectral stability (longer windows)


In [ ]:
# For debugging select each subfolder separately: 2inchStickout, 2p5inchStickout ...
# Set `INPUT_DIR` to the extracted raw data directory created by the downloader above.
RAW_EXTRACT_DIR = Path('data/raw_mat')
INPUT_DIR = RAW_EXTRACT_DIR  # adjust to a specific experiment subfolder if needed
OUTPUT_BASE = Path('../data/01_windowed_labeled_2,5s')

MAT_GLOB = '*.mat'
CHANNELS = [4, 5, 6]  # Accelerometer X,Y,Z
USE_CHANNEL = 'X'     # selected axis for labeling data

WINDOW_SEC = 2.5


# Labeling Strategy

Each time window is evaluated in both the time domain and the frequency domain to determine whether chatter-related vibration is present.

The labeling process combines:

- a time-domain amplitude criterion (vibration intensity)
- a frequency-domain harmonic criterion (chatter signature)

**NOTE**: Chatter is defined purely by harmonic spectral structure, while signal amplitude is used only as auxiliary contextual information

This reflects a practical industrial setting, where threshold-based monitoring systems are the current standard for alarm generation in condition monitoring systems.

A window is classified based on whether a chatter signature is detected:

Chatter present → chatter \
No chatter detected → no_chatter

This approach ensures that the labels remain aligned with industrial monitoring logic, while incorporating additional spectral structure for improved robustness.

## Parameter Selection for Labeling

- F_CHATTER_MAX (Frequency Cutt-Off): 
    - Chatter phenomena in machining typically manifest in low-to-mid frequency range
    - High-frequency components are more affected by sensor noise and structural resonance

- PEAK_AMP_TH (Peak Detection Threshold in Spectrum):
    - filters out low-energy spectral fluctuations and noise-induced peaks
    - Ensures that detected peaks correspond to physically meaningful vibration modes 

- TH_TIME_LOW_MIN / TH_TIME_LOW_MAX :
    - Derived from empirical distribution of vibration amplitudes across experiments
    - Separates low-energy stable cutting from higher-energy regimes

- K_MAX: defines the maximum harmonic order used to evaluate spectral consistency of chatter. It limits the search for harmonic relationships between the fundamental frequency and its integer multiples.

- F0_REL_TOL defines the allowed relative deviation when matching harmonic frequencies, scaling with the harmonic order.
- F0_ABS_TOL ensures a minimum absolute tolerance in Hz, preventing overly strict matching at low frequencies.


In [34]:
F_CHATTER_MAX = 5000.0      #  below 5 kHz
PEAK_AMP_TH = 0.001        # treshold for spectrum 0.00055

# threshold for time representation (rms)
TH_TIME_LOW_MIN = 0.02
TH_TIME_LOW_MAX = 0.06

# Harmonic-detection
K_MAX = 10                   
F0_REL_TOL = 0.02           # percentual tolerance (±2%)
F0_ABS_TOL = 5.0             # add. abs. tolerance [Hz]

# FFT window function
USE_WINDOW_FUNCTION = True  # Hann window

## Output Organisation
- extract experiment names for annotate files

In [35]:
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

experiment_name = INPUT_DIR.name
if experiment_name.startswith("timeSeries_"):
    experiment_name = experiment_name.replace("timeSeries_", "", 1)

out_dir_experiment = OUTPUT_BASE / experiment_name
out_dir_experiment.mkdir(parents=True, exist_ok=True)

mat_files = sorted(INPUT_DIR.glob(MAT_GLOB))
print("Found mat files:", len(mat_files), "experiment:", experiment_name)

Found mat files: 17 experiment: 4p5inchStickout


# Signal Processing

- **peak_abs:** Measures maximum absolute vibration amplitude within a time window
- **compute_rfft_amp:** Converts time-domain vibration signals into normalized frequency-domain amplitude spectra for harmonic analysis
- **has_harmonic_chatter:** Chatter is detected as a signal exhibiting a stable harmonic ladder structure in the frequency dominant
- **segment_time_indices:** Converts continuous vibration measurements into fixed-length segments suitable for machine learning and spectrogram generation

In [36]:
def peak_abs(x: np.ndarray) -> float:
    return float(np.max(np.abs(x)))

def compute_rfft_amp(x: np.ndarray, fs: float, use_window=True):
    N = len(x)
    if N < 2:
        raise ValueError("Segment zu kurz für FFT.")

    if use_window:
        w = np.hanning(N)
        xw = x * w
        # einfache Skalierungskorrektur für Amplituden
        win_correction = np.sum(w) / N
        win_correction = max(win_correction, 1e-12)
    else:
        xw = x
        win_correction = 1.0

    freqs = np.fft.rfftfreq(N, d=1/fs)
    fft = np.fft.rfft(xw)
    amp = np.abs(fft) / N / win_correction
    return freqs, amp

def has_harmonic_chatter(freqs, amp,
                          f_max=5000.0,
                          peak_amp_th=0.00025,
                          k_max=10,
                          f0_rel_tol=0.02,
                          f0_abs_tol=5.0):
    """
    Heuristik:
    - finde Peaks (lokale Maxima) in [0, f_max] mit amp > peak_amp_th
    - bestimme f0 als Frequenz des stärksten Peaks
    - prüfe, ob viele Harmonische bei k*f0 existieren (mit Toleranz)
    """
    # Bereich einschränken
    mask = (freqs > 0) & (freqs <= f_max)
    freqs_r = freqs[mask]
    amp_r = amp[mask]

    if len(freqs_r) == 0:
        return False, {"n_peaks": 0, "f0": np.nan, "n_harmonics_found": 0}

    # Peaks finden
    peaks, props = find_peaks(amp_r, height=peak_amp_th)
    if len(peaks) == 0:
        return False, {"n_peaks": 0, "f0": np.nan, "n_harmonics_found": 0}

    peak_freqs = freqs_r[peaks]
    peak_amps = props["peak_heights"]

    # f0 = stärkster Peak
    idx_max = int(np.argmax(peak_amps))
    f0 = float(peak_freqs[idx_max])

    # vorhandene Peak-Frequenzen (für schnelle Checks)
    peak_set = peak_freqs  # numpy array

    # prüfe Harmonische
    n_found = 0
    harmonics = []
    for k in range(2, k_max + 1):
        target = k * f0
        if target > f_max:
            break

        tol = max(f0_rel_tol * target, f0_abs_tol)
        # finde Peak, der im Toleranzband liegt
        if np.any(np.abs(peak_set - target) <= tol):
            n_found += 1
            harmonics.append(k)

    # Kriterium: mind. z.B. 2 Harmonische über Threshold -> "deutliches Rattern"
    # (kannst du später leicht anpassen)
    has = n_found >= 3

    return has, {"n_peaks": int(len(peaks)), "f0": f0, "n_harmonics_found": int(n_found), "harmonics": harmonics}

def segment_time_indices(t, fs, window_sec=5.0):
    win_len = int(window_sec * fs)
    n_windows = len(t) // win_len
    for w in range(n_windows):
        s = w * win_len
        e = s + win_len
        yield w, s, e

In [37]:
ch_idx = 0  # X-Axis for labeling

all_manifest = []

for mat_file in mat_files:
    print("Processing:", mat_file.name)

    mat = loadmat(mat_file)
    t = mat["t"][:, 0]
    d = mat["d"]

    # Samplingrate
    dt = np.median(np.diff(t))
    fs = 1.0 / dt

    # Referenzsignal für Labeling (X-Achse)
    x_label = d[:, ch_idx]

    segments = list(segment_time_indices(t, fs, window_sec=WINDOW_SEC))
    print("  segments:", len(segments))

    out_dir_mat = out_dir_experiment / mat_file.stem
    out_dir_mat.mkdir(parents=True, exist_ok=True)

    rows = []

    for w, s, e in segments:
        t_seg = t[s:e]
        x_seg = x_label[s:e]

        # =========================
        # Zeitdomäne (RMS korrekt)
        # =========================
        A_time = np.sqrt(np.mean(x_seg ** 2))

        # =========================
        # Frequenzdomäne (Labeling)
        # =========================
        freqs, amp = compute_rfft_amp(
            x_seg,
            fs,
            use_window=USE_WINDOW_FUNCTION
        )

        #mask = freqs <= F_CHATTER_MAX
        #freqs_f = freqs[mask]
        #amp_f = amp[mask]

        is_chatter, freq_info = has_harmonic_chatter(
            freqs,
            amp,
            f_max=F_CHATTER_MAX,
            peak_amp_th=PEAK_AMP_TH,
            k_max=K_MAX
        )
        # =========================
        # Labeling
        # =========================

        if is_chatter:
            label = "chatter"
        else:
            label = "no_chatter"

        # =========================
        # Speichern NPZ (3-Achsen!)
        # =========================
        seg_path = out_dir_mat / f"{mat_file.stem}_window_{w:04d}.npz"

        np.savez_compressed(
            seg_path,
            t=t_seg,
            X=d[s:e, 0],
            Y=d[s:e, 1],
            Z=d[s:e, 2],
            label=label,
            A_time=A_time,
            is_chatter=is_chatter
        )

        # =========================
        # Manifest
        # =========================
        rows.append({
            "window": w,
            "label": label,
            "is_chatter": is_chatter,
            "A_time": A_time,
            "segment_path": str(seg_path)
        })

    pd.DataFrame(rows).to_csv(
        out_dir_mat / f"{experiment_name}_label.csv",
        index=False
    )

    all_manifest.append(pd.DataFrame(rows))

print("DONE. Saved:", out_dir_experiment)

Processing: F_12-Jun-2017_rpm570_doc0p005.mat
  segments: 12
Processing: F_12-Jun-2017_rpm570_doc0p010.mat
  segments: 12
Processing: F_12-Jun-2017_rpm570_doc0p015.mat
  segments: 12
Processing: F_12-Jun-2017_rpm570_doc0p025.mat
  segments: 12
Processing: F_12-Jun-2017_rpm570_doc0p035.mat
  segments: 12
Processing: F_12-Jun-2017_rpm570_doc0p040.mat
  segments: 12
Processing: F_13-Jun-2017_rpm1030_doc0p005.mat
  segments: 4
Processing: F_13-Jun-2017_rpm1030_doc0p007.mat
  segments: 4
Processing: F_13-Jun-2017_rpm1030_doc0p010.mat
  segments: 4
Processing: F_13-Jun-2017_rpm1030_doc0p012.mat
  segments: 4
Processing: F_13-Jun-2017_rpm1030_doc0p013.mat
  segments: 4
Processing: F_13-Jun-2017_rpm1030_doc0p014.mat
  segments: 4
Processing: F_13-Jun-2017_rpm1030_doc0p015.mat
  segments: 4
Processing: F_13-Jun-2017_rpm1030_doc0p016.mat
  segments: 4
Processing: F_13-Jun-2017_rpm770_doc0p010.mat
  segments: 6
Processing: F_13-Jun-2017_rpm770_doc0p015.mat
  segments: 6
Processing: F_13-Jun-2017_